# ODP Sequence Manual Tests

这个 notebook 用来手动联调当前项目的上层时序功能，默认读取当前仓库的 `config/devices.local.yaml` 和设备 `odp_01`。

建议顺序：

1. 先运行参数与初始化单元
2. 先预览计划，再执行计划
3. 按顺序测试：单通道循环 -> 双通道同起点独立循环 -> 双通道相对时序循环 -> 双通道后上先下兼容入口
4. 每次执行后检查设备面板状态和 `runtime/sequence_logs/` 里的日志文件

注意：这里调用的是项目当前的上层服务逻辑，不是逐条手发 SCPI。执行单元会真实控制设备输出。

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "power_control_host").exists():
            return candidate
    raise RuntimeError("未找到项目根目录，请确认 notebook 位于项目工作区内。")


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

CONFIG_PATH = PROJECT_ROOT / "config" / "devices.local.yaml"
DEVICE_ID = "odp_01"
DEFAULT_LOG_DIR = PROJECT_ROOT / "runtime" / "sequence_logs"

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"CONFIG_PATH = {CONFIG_PATH}")
print(f"DEVICE_ID = {DEVICE_ID}")
print(f"DEFAULT_LOG_DIR = {DEFAULT_LOG_DIR}")

In [ ]:
from dataclasses import asdict
from datetime import datetime

try:
    import pandas as pd
except ImportError:
    pd = None

from IPython.display import display

from power_control_host.app import create_app
from power_control_host.models import ChannelCycleSpec, RelativeChannelSpec


app = create_app(CONFIG_PATH)
print("configured_devices:", app.device_service.list_devices())
print("logical_channels:", app.device_service.get_logical_channels(DEVICE_ID))


def refresh_app():
    global app
    app = create_app(CONFIG_PATH)
    print("app reloaded")
    print("configured_devices:", app.device_service.list_devices())
    print("logical_channels:", app.device_service.get_logical_channels(DEVICE_ID))
    return app


def plan_rows(plan):
    rows = []
    for index, step in enumerate(plan.steps, start=1):
        rows.append(
            {
                "step_index": index,
                "device_id": step.device_id,
                "channel": step.channel,
                "action": step.action,
                "delay_seconds": step.delay_seconds,
                "voltage": step.voltage,
                "current": step.current,
            }
        )
    return rows


def show_plan(plan):
    rows = plan_rows(plan)
    print(f"plan_name: {plan.name}")
    print(f"step_count: {len(rows)}")
    if pd is not None:
        display(pd.DataFrame(rows))
    else:
        for row in rows:
            print(row)
    return rows


def event_rows(events):
    return [asdict(item) for item in events]


def make_log_path(label: str) -> Path:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return DEFAULT_LOG_DIR / f"{label}_{timestamp}.csv"


def run_plan(plan, label: str | None = None):
    log_path = make_log_path(label or plan.name)
    events = app.sequence_service.execute_plan(plan, log_path=log_path)
    print(f"executed_plan: {plan.name}")
    print(f"event_count: {len(events)}")
    print(f"log_path: {log_path}")
    rows = event_rows(events)
    if pd is not None:
        display(pd.DataFrame(rows))
    else:
        for row in rows:
            print(row)
    return events, log_path


def measure(channel: str):
    sample = app.device_service.read_measurement(DEVICE_ID, channel)
    payload = asdict(sample)
    for key, value in payload.items():
        print(f"{key}: {value}")
    return sample

## 1. 单通道循环

这个块对应 CLI 的 `run-cycle`。先预览，再执行。

In [ ]:
single_channel = "CH1"
single_on_seconds = 3.0
single_off_seconds = 2.0
single_cycles = 3
single_voltage = 5.0
single_current = 0.5
single_log_label = "manual_single_cycle"

In [ ]:
single_plan = app.sequence_service.build_single_channel_cycle_plan(
    device_id=DEVICE_ID,
    channel=single_channel,
    on_seconds=single_on_seconds,
    off_seconds=single_off_seconds,
    cycles=single_cycles,
    voltage=single_voltage,
    current=single_current,
)
show_plan(single_plan)

In [ ]:
single_events, single_log_path = run_plan(single_plan, single_log_label)

In [ ]:
measure(single_channel)

## 2. 双通道同起点独立循环

这个块对应 CLI 的 `run-parallel-cycle`。两个通道从同一起点开始，但各自按自己的循环参数运行。

In [ ]:
parallel_specs = [
    ChannelCycleSpec(channel="CH1", on_seconds=6.0, off_seconds=3.0, cycles=2, voltage=5.0, current=0.5),
    ChannelCycleSpec(channel="CH2", on_seconds=4.0, off_seconds=2.0, cycles=2, voltage=3.3, current=0.3),
]
parallel_log_label = "manual_parallel_cycle"

In [ ]:
parallel_plan = app.sequence_service.build_parallel_channel_cycle_plan(
    device_id=DEVICE_ID,
    channel_specs=parallel_specs,
)
show_plan(parallel_plan)

In [ ]:
parallel_events, parallel_log_path = run_plan(parallel_plan, parallel_log_label)

## 3. 双通道相对时序循环

这个块对应 CLI 的 `run-relative-cycle`。`CH1` 作为基准通道，`CH2` 相对 `CH1` 延后上电、提前下电。

In [ ]:
relative_on_seconds = 10.0
relative_off_seconds = 3.0
relative_cycles = 2
relative_specs = [
    RelativeChannelSpec(channel="CH1", voltage=5.0, current=0.5),
    RelativeChannelSpec(
        channel="CH2",
        reference_channel="CH1",
        on_delay_seconds=2.0,
        off_advance_seconds=2.0,
        voltage=3.3,
        current=0.3,
    ),
]
relative_log_label = "manual_relative_cycle"

In [ ]:
relative_plan = app.sequence_service.build_relative_channel_cycle_plan(
    device_id=DEVICE_ID,
    on_seconds=relative_on_seconds,
    off_seconds=relative_off_seconds,
    cycles=relative_cycles,
    channel_specs=relative_specs,
)
show_plan(relative_plan)

In [ ]:
relative_events, relative_log_path = run_plan(relative_plan, relative_log_label)

## 4. 双通道后上先下兼容入口

这个块对应 CLI 的 `run-staggered-cycle`，本质上是双通道相对时序循环的兼容写法。

In [ ]:
staggered_lead_channel = "CH1"
staggered_lag_channel = "CH2"
staggered_delay_seconds = 2.0
staggered_hold_seconds = 4.0
staggered_rest_seconds = 3.0
staggered_cycles = 2
staggered_lead_voltage = 5.0
staggered_lead_current = 0.5
staggered_lag_voltage = 3.3
staggered_lag_current = 0.3
staggered_log_label = "manual_staggered_cycle"

In [ ]:
staggered_plan = app.sequence_service.build_staggered_channel_cycle_plan(
    device_id=DEVICE_ID,
    lead_channel=staggered_lead_channel,
    lag_channel=staggered_lag_channel,
    delay_seconds=staggered_delay_seconds,
    hold_seconds=staggered_hold_seconds,
    rest_seconds=staggered_rest_seconds,
    cycles=staggered_cycles,
    lead_voltage=staggered_lead_voltage,
    lead_current=staggered_lead_current,
    lag_voltage=staggered_lag_voltage,
    lag_current=staggered_lag_current,
)
show_plan(staggered_plan)

In [ ]:
staggered_events, staggered_log_path = run_plan(staggered_plan, staggered_log_label)

## 5. 补充说明

- 如果你改了 `config/devices.local.yaml`，请重新运行初始化单元，或者执行 `refresh_app()`。
- 如果只想验证某个块，直接改对应参数单元，然后只执行该块的预览和执行单元即可。
- 执行完成后，日志会写到 `runtime/sequence_logs/`。